In [38]:
# Import libraries
import pandas as pd
import sys
from pathlib import Path
from datetime import datetime

In [2]:
# Set Paths
BASE_PATH = Path().resolve()
RAW_PATH = BASE_PATH.parent.parent / 'data' / 'raw'
RAW_FILE = RAW_PATH / 'test_sample_data_customers.xlsx'
CLEAN_FILE = 'customers_insert_test.sql'
LIB_PATH = BASE_PATH.parent / 'python-scripts'

In [3]:
# Set Custom Libraries
sys.path.append(str(LIB_PATH))
from helper_functions import HelperFunctions

In [4]:
# Instace objects
helper = HelperFunctions()

### Manejo de Datos

Realizar una limpieza del archivo _test_sample_data_customers.xlsx_
transformando los datos, en clientes y direcciones, que se puedan
insertar en la tabla: `cs.customers` y `cs.addresses`

In [47]:
# 01. Read file
df_raw = pd.read_excel(RAW_FILE)
print(f'Shape: {df_raw.shape}')
display(df_raw.head())

Shape: (4, 10)


,TIPO DE DOCUMENTO,NUMERO DE DOCUMENTO,PRIMER APELLIDO,SEGUNDO APELLIDO,PRIMER NOMBRE,SEGUNDO NOMBRE,BARRIO,DIRECCION,TELEFONO PRINCIPAL,TELEFONO SECUNDARIO
0,PPT,1098458789,PEREZ,CASTRO,JUAN,NaN,KENN-CASA LOMA,KR 80J 71 15 SUR,3118980243,NaN
1,CC,10056987456,FERMIN,LOPEZ,GABRIEL,MATEO,BOSA-SAN PABLO,KR 5B 7 43 ESTE,3214337211,3.025900e+08
2,TI,10037897123,JAIMES,NaN,RAUL,ERNESTO,ENGATIVA-SALOME,KR 25-219 MZ 2714,3103066943,NaN
3,CC,56987456,TORREZ,GELVEZ,LUISA,DANIELA,FONTIBON-HAYUELOS,KR 96B – 448A,301258978,3.112775e+09


In [48]:
# 02. Clean columns names
old_columns = df_raw.columns
# apply rename columns
df_t1 = df_raw.copy()
df_t1.rename(columns=helper.rename_cols, inplace=True)
new_columns = df_t1.columns
# Show
print(f"Old Columns: {old_columns}\nNew Columns: {new_columns}\n")
display(df_t1.head())


Old Columns: Index(['TIPO DE DOCUMENTO', 'NUMERO DE DOCUMENTO', 'PRIMER APELLIDO',
       'SEGUNDO APELLIDO', 'PRIMER NOMBRE', 'SEGUNDO NOMBRE', 'BARRIO',
       'DIRECCION', 'TELEFONO PRINCIPAL', 'TELEFONO SECUNDARIO'],
      dtype='str')
New Columns: Index(['tipo_de_documento', 'numero_de_documento', 'primer_apellido',
       'segundo_apellido', 'primer_nombre', 'segundo_nombre', 'barrio',
       'direccion', 'telefono_principal', 'telefono_secundario'],
      dtype='str')



,tipo_de_documento,numero_de_documento,primer_apellido,segundo_apellido,primer_nombre,segundo_nombre,barrio,direccion,telefono_principal,telefono_secundario
0,PPT,1098458789,PEREZ,CASTRO,JUAN,NaN,KENN-CASA LOMA,KR 80J 71 15 SUR,3118980243,NaN
1,CC,10056987456,FERMIN,LOPEZ,GABRIEL,MATEO,BOSA-SAN PABLO,KR 5B 7 43 ESTE,3214337211,3.025900e+08
2,TI,10037897123,JAIMES,NaN,RAUL,ERNESTO,ENGATIVA-SALOME,KR 25-219 MZ 2714,3103066943,NaN
3,CC,56987456,TORREZ,GELVEZ,LUISA,DANIELA,FONTIBON-HAYUELOS,KR 96B – 448A,301258978,3.112775e+09


#### CS.CUSTOMERS

Sentencia de insercion en `cs.customers`
```sql
INSERT INTO cs.customers (id_number, birth_date, phone_number, name, email, created_at, document_type_code) VALUES 
('1280058', '2009-05-29', '3329492370', 'NINO PINILLA OCTAVIANO', 'nino.pinilla38@central.io', '2026-03-08 03:10:41.217547894', '1');
```

In [49]:
# START FIELD BY FIELD
df_t2 = df_t1.copy()

In [50]:
# columns cs.customers

# birth_date
birthday_dates = ['1965-05-18', '1999-01-30', '2005-07-25', '2000-08-13']
df_t2['birth_date'] = birthday_dates

In [51]:
# name: primer_nombre + segundo_nombre + primer_apellido + segundo_apellido
df_t2['name'] = (df_t2['primer_nombre'] + ' ' + 
                df_t2['segundo_nombre'].fillna('') + ' ' +
                df_t2['primer_apellido'] + ' ' +
                df_t2['segundo_apellido'].fillna('')
)

In [52]:
# email
df_t2['email'] =  df_t2['name'].apply(helper.create_email)

In [53]:
# document_type_code
# Create a function to mapping the document number (str) to code.
documents_types = {
    "CC" : "1",
    "TI" : "4",
    "PPT" : "10"
}

df_t2['document_type_code'] = (
    df_t2['tipo_de_documento'].astype(str) \
        .str.strip() \
            .map(documents_types)
)

In [54]:
# phone_number: convert to string
df_t2['telefono_principal'] = df_t2['telefono_principal'].astype(str)

In [55]:
df_t2.loc[:, ['tipo_de_documento', 'document_type_code']]

,tipo_de_documento,document_type_code
0,PPT,10
1,CC,1
2,TI,4
3,CC,1


In [56]:
# map columns
map_columns = {'numero_de_documento' : 'id_number', 
               'telefono_principal' : 'phone_number'}
df_t2.rename(columns=map_columns, inplace=True)

In [57]:
# Select columns
# id_number, birth_date, phone_number, name, email, created_at, document_type_code)
cols = ['id_number', 'birth_date', 'phone_number', 'name' , 'email', 'document_type_code']
df_customers = helper.selected_cols(df_t2, cols)

display(df_customers.head())

,id_number,birth_date,phone_number,name,email,document_type_code
0,1098458789,1965-05-18,3118980243,JUAN PEREZ CASTRO,juan.42@hotmail.com,10
1,10056987456,1999-01-30,3214337211,GABRIEL MATEO FERMIN LOPEZ,gabriel.mateo@outlook.com,1
2,10037897123,2005-07-25,3103066943,RAUL ERNESTO JAIMES,raul.ernesto@outlook.com,4
3,56987456,2000-08-13,301258978,LUISA DANIELA TORREZ GELVEZ,luisa.daniela46@popular.com,1


## Save Data

In [58]:
# Current date and set path save
current = datetime.now().strftime("%Y%m%d")
SAVE_ = Path().resolve()
CLEAN_PATH = BASE_PATH.parent.parent / 'data' / 'clean'

In [59]:
# Customers
sql_customers_statement = helper.generate_sql_statements(
    df = df_customers,
    schema = 'cs',
    table_name = 'customers'
)
# Save path
filename = CLEAN_PATH / 'customers_sample_data.sql'
helper.save_sql_statement(sql_customers_statement, filename)

SQL statement saved to /home/alejocjaimes/Documents/Github/Doc-UP-AlejandroJaimes/sql-101-mastering/content/customers/data/clean/customers_sample_data.sql
